In [1]:
!pip install datasets

In [2]:
!pip install "accelerate>=0.26.0"

In [3]:
import json
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import CosineSimilarityLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments




def load_hf_dataset(path="train.json"):
  with open(path, "r", encoding="utf-8") as f:
    rows = json.load(f)["data"]


  # HuggingFace Dataset 형식으로 변환
  return Dataset.from_list(
    [{"sentence1": r["text1"], "sentence2": r["text2"], "label": float(r["label"])} for r in rows]
  )




def main():
  base_model = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
  model = SentenceTransformer(base_model)


  train_ds = load_hf_dataset("train.json")


  loss = CosineSimilarityLoss(model)


  args = SentenceTransformerTrainingArguments(
    output_dir="sbert_specup_model",
    num_train_epochs=1,         # 처음엔 1로
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=False,                 # GPU 없으면 False
    logging_steps=50,
    save_strategy="no"
  )


  trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    loss=loss
  )


  trainer.train()


  model.save("sbert_specup_model")
  print("Saved model to: sbert_specup_model")


##

if __name__ == "__main__":
  main()


c:\Users\KOSMO\miniconda3\envs\myvenv2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\KOSMO\miniconda3\envs\myvenv2\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.037400
100,0.021200
150,0.016000
200,0.014700
250,0.014700
300,0.014100
350,0.014600
400,0.012300
450,0.009800
500,0.008600


Saved model to: sbert_specup_model


In [1]:
!pip install sentence-transformers scikit-learn matplotlib pandas numpy

In [9]:
import json
import math
import numpy as np

from collections import defaultdict
from sentence_transformers import SentenceTransformer, util


MODEL_PATH = "./sbert_specup_model"
TEST_FILE = "test.json"

# label이 이 값 이상이면 "추천 정답"으로 간주
RELEVANCE_THRESHOLD = 0.5

# 평가할 K들
K_LIST = [1, 3, 5, 10]


model = SentenceTransformer(MODEL_PATH)


def cosine_similarity(a, b):
  emb1 = model.encode(a, convert_to_tensor=True)
  emb2 = model.encode(b, convert_to_tensor=True)
  return float(util.cos_sim(emb1, emb2).item())


def precision_at_k(binary_relevance, k):
  top_k = binary_relevance[:k]
  if k == 0:
    return 0.0
  return sum(top_k) / k


def recall_at_k(binary_relevance, total_relevant, k):
  if total_relevant == 0:
    return 0.0
  top_k = binary_relevance[:k]
  return sum(top_k) / total_relevant


def dcg_at_k(relevance_scores, k):
  dcg = 0.0
  for i, rel in enumerate(relevance_scores[:k]):
    dcg += (2 ** rel - 1) / math.log2(i + 2)
  return dcg


def ndcg_at_k(true_relevance_sorted_by_pred, true_relevance_ideal, k):
  dcg = dcg_at_k(true_relevance_sorted_by_pred, k)
  idcg = dcg_at_k(true_relevance_ideal, k)
  if idcg == 0:
    return 0.0
  return dcg / idcg


def average_precision(binary_relevance):
  hit_count = 0
  precision_sum = 0.0

  for i, rel in enumerate(binary_relevance, start=1):
    if rel == 1:
      hit_count += 1
      precision_sum += hit_count / i

  if hit_count == 0:
    return 0.0

  return precision_sum / hit_count


with open(TEST_FILE, "r", encoding="utf-8") as f:
  raw = json.load(f)

data = raw["data"]


grouped = defaultdict(list)
for row in data:
  grouped[row["user_id"]].append(row)


all_results = {k: {"precision": [], "recall": [], "ndcg": []} for k in K_LIST}
map_scores = []

print("\n===== 사용자별 추천 평가 =====")

for user_id, rows in grouped.items():
  scored_items = []

  for row in rows:
    pred_score = cosine_similarity(row["text1"], row["text2"])
    true_score = float(row["label"])

    scored_items.append({
      "posting_id": row["posting_id"],
      "pred_score": pred_score,
      "true_score": true_score,
      "binary_label": 1 if true_score >= RELEVANCE_THRESHOLD else 0
    })

  pred_ranked = sorted(scored_items, key=lambda x: x["pred_score"], reverse=True)
  ideal_ranked = sorted(scored_items, key=lambda x: x["true_score"], reverse=True)

  binary_relevance = [item["binary_label"] for item in pred_ranked]
  total_relevant = sum(binary_relevance)

  true_relevance_sorted_by_pred = [item["true_score"] for item in pred_ranked]
  true_relevance_ideal = [item["true_score"] for item in ideal_ranked]

  ap = average_precision(binary_relevance)
  map_scores.append(ap)

  print(f"\n--- user_id: {user_id}")
  print("예측 순위 Top 10")
  for rank, item in enumerate(pred_ranked[:10], start=1):
    print(
      f"{rank:2d}. posting_id={item['posting_id']}, "
      f"pred={item['pred_score']:.4f}, "
      f"true={item['true_score']:.4f}, "
      f"relevant={item['binary_label']}"
    )

  for k in K_LIST:
    p_k = precision_at_k(binary_relevance, k)
    r_k = recall_at_k(binary_relevance, total_relevant, k)
    n_k = ndcg_at_k(true_relevance_sorted_by_pred, true_relevance_ideal, k)

    all_results[k]["precision"].append(p_k)
    all_results[k]["recall"].append(r_k)
    all_results[k]["ndcg"].append(n_k)

    print(
      f"P@{k}={p_k:.4f}, "
      f"R@{k}={r_k:.4f}, "
      f"NDCG@{k}={n_k:.4f}"
    )

  print(f"AP={ap:.4f}")


print("\n===== 전체 추천 성능 평균 =====")

for k in K_LIST:
  mean_p = np.mean(all_results[k]["precision"]) if all_results[k]["precision"] else 0.0
  mean_r = np.mean(all_results[k]["recall"]) if all_results[k]["recall"] else 0.0
  mean_n = np.mean(all_results[k]["ndcg"]) if all_results[k]["ndcg"] else 0.0

  print(
    f"Precision@{k}: {mean_p:.4f} | "
    f"Recall@{k}: {mean_r:.4f} | "
    f"NDCG@{k}: {mean_n:.4f}"
  )

mean_map = np.mean(map_scores) if map_scores else 0.0
print(f"MAP: {mean_map:.4f}")

The tokenizer you are loading from './sbert_specup_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



===== 사용자별 추천 평가 =====

--- user_id: test_make_user001
예측 순위 Top 10
 1. posting_id=53108021, pred=0.5663, true=0.5204, relevant=1
 2. posting_id=52987701, pred=0.4871, true=0.5333, relevant=1
 3. posting_id=53049578, pred=0.4700, true=0.7765, relevant=1
 4. posting_id=52923191, pred=0.4330, true=0.5449, relevant=1
 5. posting_id=52931188, pred=0.4280, true=0.5449, relevant=1
 6. posting_id=53127763, pred=0.3953, true=0.1717, relevant=0
 7. posting_id=53127742, pred=0.3953, true=0.1717, relevant=0
 8. posting_id=52985175, pred=0.3785, true=0.4940, relevant=0
 9. posting_id=53112307, pred=0.3705, true=0.1894, relevant=0
10. posting_id=52964466, pred=0.2737, true=0.2459, relevant=0
P@1=1.0000, R@1=0.2000, NDCG@1=0.6092
P@3=1.0000, R@3=0.6000, NDCG@3=0.8710
P@5=1.0000, R@5=1.0000, NDCG@5=0.9093
P@10=0.5000, R@10=1.0000, NDCG@10=0.9173
AP=1.0000

--- user_id: test_make_user002
예측 순위 Top 10
 1. posting_id=52968999, pred=0.5114, true=0.3423, relevant=0
 2. posting_id=52968989, pred=0.5114, t